# Day 4: HuggingFace Transformers - Solutions

**Warning:** Try to complete the exercises in `demos/07_HuggingFace_Transformers.ipynb` on your own before looking at these solutions!

---

## Setup

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load pipelines (these are reused across exercises)
sentiment_analyzer = pipeline("sentiment-analysis")
classifier = pipeline("zero-shot-classification")

print("✓ All pipelines loaded!")

---

## Exercise 1: Customer Review Analyzer

### Solution: Analyze reviews, display results, and visualize

In [ ]:
# Sample reviews
reviews = [
    "The product arrived quickly and works perfectly. Very happy with my purchase!",
    "Terrible quality. Broke after just two days of use.",
    "Good value for the price. Not amazing but does the job.",
    "Absolutely love it! Best purchase I've made this year.",
    "The color is different from what was shown. Disappointed.",
    "Shipping was slow but the product itself is great.",
    "Would not recommend. Wasted my money.",
    "Exceeded my expectations! Will buy again."
]

# Solution: Analyze all reviews with the sentiment pipeline
review_results = sentiment_analyzer(reviews)

# Solution: Print results in a formatted table
print(f"{'Review':<65} {'Sentiment':>10} {'Confidence':>12}")
print("=" * 90)
for review, result in zip(reviews, review_results):
    short = review[:60] + '...' if len(review) > 60 else review
    print(f"{short:<65} {result['label']:>10} {result['score']:>11.1%}")

# Solution: Calculate overall stats
num_positive = sum(1 for r in review_results if r['label'] == 'POSITIVE')
num_negative = sum(1 for r in review_results if r['label'] == 'NEGATIVE')
avg_confidence = np.mean([r['score'] for r in review_results])

print(f"\n--- Summary ---")
print(f"Positive reviews: {num_positive}/{len(reviews)}")
print(f"Negative reviews: {num_negative}/{len(reviews)}")
print(f"Average confidence: {avg_confidence:.1%}")

**Explanation:**
- `sentiment_analyzer(reviews)` accepts a list and returns results for all items at once (batched inference)
- Each result is a dictionary with `'label'` (POSITIVE/NEGATIVE) and `'score'` (confidence 0–1)
- We use a list comprehension with `sum()` to count labels and `np.mean()` for the average confidence

In [ ]:
# Solution: Visualize the results
scores = [
    r['score'] if r['label'] == 'POSITIVE' else -r['score']
    for r in review_results
]
colors = ['green' if s > 0 else 'red' for s in scores]
short_reviews = [r[:40] + '...' if len(r) > 40 else r for r in reviews]

plt.figure(figsize=(12, 6))
plt.barh(range(len(reviews)), scores, color=colors, alpha=0.7, edgecolor='black')
plt.yticks(range(len(reviews)), short_reviews, fontsize=10)
plt.xlabel('Sentiment Score (negative ← → positive)', fontsize=12)
plt.title('Customer Review Sentiment Analysis', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.xlim([-1.1, 1.1])
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

**Explanation:**
- We convert scores to signed values: positive stays positive, negative becomes negative (e.g., 0.95 NEGATIVE → -0.95)
- This makes the bar chart intuitive: bars go right for positive sentiment, left for negative
- `plt.barh()` creates horizontal bars, which work well for text labels
- We truncate review text to 40 characters so labels fit on the y-axis

---

## Exercise 2: News Article Classifier

### Solution: Classify headlines with custom categories

In [ ]:
# Solution: Define headlines and categories
headlines = [
    "New study finds regular exercise reduces risk of heart disease by 30%",
    "Tech giant announces record quarterly earnings beating all expectations",
    "World leaders meet at UN summit to discuss climate change agreements",
    "Local team wins national championship in thrilling overtime final",
    "Scientists develop new solar panel technology that doubles energy output"
]

categories = ["politics", "sports", "technology", "health", "business", "science", "environment"]

# Solution: Classify each headline and display results
print(f"{'Headline':<70} {'Top Category':>14} {'Confidence':>12}")
print("=" * 100)

for headline in headlines:
    result = classifier(headline, categories)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    short = headline[:65] + '...' if len(headline) > 65 else headline
    print(f"{short:<70} {top_label:>14} {top_score:>11.1%}")

In [ ]:
# Bonus: Show full classification breakdown for one headline
example_headline = headlines[2]  # The climate/politics headline
result = classifier(example_headline, categories)

print(f"Headline: '{example_headline}'\n")
print(f"{'Category':<15} {'Confidence':>10}  Bar")
print("-" * 50)
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 40)
    print(f"{label:<15} {score:>9.1%}  {bar}")

**Explanation:**
- `classifier(text, candidate_labels)` returns all labels sorted by descending confidence
- `result['labels'][0]` is the top prediction, `result['scores'][0]` its confidence
- Zero-shot classification works because the model understands the *meaning* of both the text and the category words
- Notice how some headlines match multiple categories (e.g., climate change could be "environment" or "politics") — the model picks the most likely one
- Adding more specific categories (like "science" and "environment") gives the model finer-grained options to choose from

---

## Exercise 3: Explore a HuggingFace Dataset

### Solution: Load and explore the `rotten_tomatoes` dataset, then apply sentiment analysis

In [ ]:
# Solution: Load a dataset
my_dataset = load_dataset("rotten_tomatoes")

# Solution: Explore the dataset structure
print("Dataset structure:")
print(my_dataset)
print(f"\nFeatures: {my_dataset['train'].features}")
print(f"\nNumber of examples:")
for split in my_dataset:
    print(f"  {split}: {len(my_dataset[split]):,}")

In [ ]:
# Solution: Display first few examples
print("First 5 examples:\n")
for i in range(5):
    example = my_dataset['train'][i]
    label_str = 'positive' if example['label'] == 1 else 'negative'
    print(f"[{label_str:>8}] {example['text'][:80]}...")
    print()

In [ ]:
# Solution: Convert to Pandas and show statistics
rt_df = my_dataset['train'].to_pandas()

print("DataFrame shape:", rt_df.shape)
print("\nFirst rows:")
print(rt_df.head())

# Label distribution
print(f"\nLabel distribution:")
print(rt_df['label'].value_counts().rename({0: 'negative', 1: 'positive'}))

# Review length statistics
rt_df['word_count'] = rt_df['text'].str.split().str.len()
print(f"\nReview length statistics:")
print(f"  Min: {rt_df['word_count'].min()} words")
print(f"  Max: {rt_df['word_count'].max()} words")
print(f"  Mean: {rt_df['word_count'].mean():.0f} words")
print(f"  Median: {rt_df['word_count'].median():.0f} words")

In [ ]:
# Solution: Apply sentiment analysis to a sample
sample_size = 10
sample = my_dataset['test'][:sample_size]

predictions = sentiment_analyzer(sample['text'])

# Compare predictions with true labels
correct = 0
print(f"{'True':<10} {'Predicted':<12} {'Confidence':>10}   Review Preview")
print("=" * 85)
for true_label, pred, text in zip(sample['label'], predictions, sample['text']):
    true_str = 'positive' if true_label == 1 else 'negative'
    pred_str = pred['label'].lower()
    match = '✓' if true_str == pred_str else '✗'
    if true_str == pred_str:
        correct += 1
    preview = text[:45].replace('\n', ' ') + '...'
    print(f"{true_str:<10} {pred_str:<12} {pred['score']:>9.1%}  {match} {preview}")

print(f"\nAccuracy on sample: {correct}/{sample_size} ({correct/sample_size:.0%})")

**Explanation:**
- `load_dataset("rotten_tomatoes")` downloads the Rotten Tomatoes movie review dataset with train/validation/test splits
- `.to_pandas()` converts a HuggingFace dataset split into a familiar Pandas DataFrame
- We compute basic text statistics using Pandas string methods (`str.split().str.len()`)
- Applying `sentiment_analyzer` to the test set lets us check how well the pre-trained model performs on this data *without any training*
- The pre-trained sentiment model typically performs reasonably well on movie reviews, since it was trained on similar data (SST-2 dataset)

---

## Summary

In these exercises you practiced:

✓ Using `sentiment_analyzer()` on a batch of texts and computing summary statistics  
✓ Visualizing sentiment results with a horizontal bar chart  
✓ Defining custom categories for zero-shot classification  
✓ Loading a HuggingFace dataset, exploring it with Pandas, and applying a pipeline  

**Key takeaway:** HuggingFace pipelines let you apply powerful pre-trained models with just a few lines of code — no training required!

---